# AlphaTrain Pillar 2X: V10 policy-only training

First training run that uses the new `--policy-only` flag — drops the dead value head from the architecture (HISTORY lessons 116-118).

**Data:** V10 self-play with feature-value MCTS, **5.77M states** from:
- 932 regular games @ 800 sims, cap 6000 turns (~3.66M states, mean 8,234 / median 9,328 / 40% capped)
- 2,226 opening games @ 800 sims, cap 500 turns (~1.07M states, mean 982 — early-game diversity)
- 5,972 crisis games @ 800 sims with recovery-15/prevention-30 rewinds (~1.00M states, mean 3,512 — 60% capped)

Built with `build_expert_v2_tensor.py --policy-only-data` (V10 writes bootstrap_value=0 for capped games — the builder requires this flag to acknowledge that those targets are unused at training time).

**Model:** Warm start from pillar2w2 epoch 10. Backbone + policy head weights transfer (139 keys); the 10 value-head keys are filtered out automatically because `--policy-only` builds an architecture without them.

**Upload to Drive (`MyDrive/alphatrain/`):**
1. `colorlines_selfplay_train.tar.gz` — code (~67 KB)
2. `selfplay_v10.pt.gz` — V10 tensor (~440 MB compressed, ~3.4 GB unpacked)
3. `pillar2w2_epoch_10.pt` — warm-start weights

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, shutil, time

DRIVE = '/content/drive/MyDrive/alphatrain'

!cp {DRIVE}/colorlines_selfplay_train.tar.gz /content/
!cd /content && tar xzf colorlines_selfplay_train.tar.gz

os.makedirs('/content/alphatrain/data', exist_ok=True)
shutil.copy(f'{DRIVE}/pillar2w2_epoch_10.pt', '/content/alphatrain/data/model.pt')
print(f'Model: {os.path.getsize("/content/alphatrain/data/model.pt")/1e6:.0f} MB')

t0 = time.time()
!gzip -dc {DRIVE}/selfplay_v10.pt.gz > /content/alphatrain/data/selfplay.pt
print(f'Data: {os.path.getsize("/content/alphatrain/data/selfplay.pt")/1e9:.1f} GB ({time.time()-t0:.0f}s)')

!pip install -q numpy numba scipy pytest

In [ ]:
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    g = torch.cuda.get_device_properties(0)
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {g.total_memory / 1e9:.1f} GB')

!cd /content && python -m pytest alphatrain/tests/test_model.py -v --tb=short 2>&1 | tail -5

In [ ]:
# Pillar 2X: V10 data (5.77M states), warm-start from 2W2 ep10, policy-only.
# --policy-only drops the value head; warm-start loader filters value_*
# keys automatically since the architecture no longer has them.
%cd /content
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python -m alphatrain.train \
    --tensor-file alphatrain/data/selfplay.pt \
    --amp --compile \
    --resume alphatrain/data/model.pt --warm-start \
    --policy-only \
    --epochs 10 --batch-size 32768 --lr 1e-4 --warmup-epochs 2 \
    --copy-to /content/drive/MyDrive/alphatrain/pillar2x_best.pt \
    --save-dir /content/checkpoints/pillar2x

In [ ]:
# Copy ALL epoch checkpoints to Drive
import shutil, os, glob
DRIVE = '/content/drive/MyDrive/alphatrain'
for f in sorted(glob.glob('/content/checkpoints/pillar2x/epoch_*.pt')):
    dst = f'{DRIVE}/pillar2x_{os.path.basename(f)}'
    shutil.copy(f, dst)
    print(f'Saved {dst} ({os.path.getsize(dst)/1e6:.0f} MB)')
for f in ['best.pt', 'latest.pt']:
    src = f'/content/checkpoints/pillar2x/{f}'
    if os.path.exists(src):
        shutil.copy(src, f'{DRIVE}/pillar2x_{f}')